# bench_stats

> Statistics for the Phase 8 benchmark: bootstrap intervals, sign-flip permutation tests, Holm correction, paired effect sizes and scaling fits (numpy only).

Spec: `specs/multiagent/phase8/` (design-low-level section 3).

In [ ]:
#| default_exp bench_stats

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import numpy as np

NAN = float('nan')


def _clean(x):
    x = np.asarray(x, float)
    return x[~np.isnan(x)]


def bootCI(x, n=2000, seed=0):
    "95% percentile bootstrap interval of the mean (nan entries dropped)."
    x = _clean(x)
    if not len(x):
        return NAN, NAN
    m = x[np.random.default_rng(seed).integers(0, len(x), size=(n, len(x)))].mean(1)
    return float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))


def signFlipP(d, n=10000, seed=0):
    "Two-sided sign-flip p-value of a paired mean difference: exact when 2^len(d) <= n, else seeded random flips."
    d = _clean(d)
    if not len(d):
        return NAN
    obs, m = abs(d.mean()), len(d)
    if obs == 0:
        return 1.0
    if 2 ** m <= n:
        signs = 1 - 2 * ((np.arange(2 ** m)[:, None] >> np.arange(m)) & 1)
        return float((np.abs((signs * d).mean(1)) >= obs - 1e-12).mean())
    s = np.random.default_rng(seed).choice([-1.0, 1.0], size=(n, m))
    return float((1 + (np.abs((s * d).mean(1)) >= obs - 1e-12).sum()) / (1 + n))


def holm(p):
    "Holm-adjusted p-values, in the input order."
    p = np.asarray(p, float)
    adj, run = np.empty(len(p)), 0.0
    for r, i in enumerate(np.argsort(p, kind='stable')):
        run = max(run, min(1.0, (len(p) - r) * p[i]))
        adj[i] = run
    return adj


def cohenDz(d):
    "Paired effect size: mean over standard deviation (ddof 1) of the differences."
    d = _clean(d)
    return float(d.mean() / d.std(ddof=1)) if len(d) > 1 and d.std(ddof=1) > 0 else NAN


def scalingFit(N, y, seeds, n=2000, seed=0):
    "OLS of y on log2 N; slope CI by resampling seeds (all sizes of a seed together)."
    x, y, seeds = np.log2(np.asarray(N, float)), np.asarray(y, float), np.asarray(seeds)
    ok = ~np.isnan(y)
    x, y, seeds = x[ok], y[ok], seeds[ok]
    slope, icpt = np.polyfit(x, y, 1)
    groups = [np.flatnonzero(seeds == s) for s in np.unique(seeds)]
    rng, boots = np.random.default_rng(seed), []
    for _ in range(n):
        idx = np.concatenate([groups[k] for k in rng.integers(0, len(groups), len(groups))])
        if len(np.unique(x[idx])) > 1:
            boots.append(np.polyfit(x[idx], y[idx], 1)[0])
    return dict(slope=float(slope), intercept=float(icpt),
                ci=(float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))) if boots else (NAN, NAN))

## Tests

In [ ]:
import itertools
import numpy as np
from fastcore.test import test_eq, test_close
from explRL.bench_stats import bootCI, signFlipP, holm, cohenDz, scalingFit

test_close(bootCI([0.3] * 30), (0.3, 0.3))
lo, hi = bootCI(np.arange(30.0)); assert lo < 14.5 < hi
test_eq(signFlipP(np.zeros(30)), 1.0)
d = np.array([0.5, 1.2, -0.3, 0.8, 0.9, 1.1, -0.2, 0.7])
exact = np.mean([abs(np.mean(np.array(s) * d)) >= abs(d.mean()) - 1e-12 for s in itertools.product([1, -1], repeat=8)])
test_close(signFlipP(d), exact)
d30 = np.random.default_rng(1).normal(0.5, 1.0, 30)
assert signFlipP(d30) < 0.05 and signFlipP(d30) == signFlipP(d30)     # significant, and deterministic
test_close(holm([0.01, 0.04, 0.03]), [0.03, 0.06, 0.06])
test_close(holm([0.5, 0.001]), [0.5, 0.002])
test_close(cohenDz([1.0, 2.0, 3.0]), 2.0)
assert np.isnan(cohenDz([1.0, 1.0]))
N = np.tile([1, 2, 4, 8, 16, 32, 64], 5); seeds = np.repeat(np.arange(5), 7)
f = scalingFit(N, 0.2 + 0.1 * np.log2(N), seeds)
test_close((f['slope'], f['intercept']), (0.1, 0.2)); test_close(f['ci'], (0.1, 0.1))

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()